# 02 - Hybrid Resampling & CTGAN

Notebook này thực hiện trạm cân bằng dữ liệu:
1. Nạp `01_processed_features.parquet`
2. Tách majority/minority
3. Undersampling với luật M, 2.5M, transition x1.30, duplicate x0.20
4. CTGAN sinh thêm lớp 4, 5
5. Sanity check vật lý
6. Merge + shuffle + export `02_balanced_training_data.parquet`

In [ ]:
import os
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 180)

In [ ]:
# ==== Config ====
INPUT_PATH = '/workspace/ai-core/notebooks/01_processed_features.parquet'
OUTPUT_PATH = '/workspace/ai-core/notebooks/02_balanced_training_data.parquet'

SEED = 42
TARGET_COL = 'congestion_level'
ANCHOR_CLASS = 3
MAJORITY_CLASSES = [0, 1, 2]
MINORITY_CLASSES = [4, 5]

TRANSITION_MULTIPLIER = 1.30
DUPLICATE_MULTIPLIER = 0.20
DUPLICATE_MAE_THRESHOLD = 1e-3
CAP_MULTIPLIER = 2.5

# Synthetic targets
TARGET_SYNTHETIC_CLASS_4 = 50_000
TARGET_SYNTHETIC_CLASS_5 = 20_000

CTGAN_EPOCHS = 20
USE_CTGAN = True

rng = np.random.default_rng(SEED)

In [ ]:
df = pd.read_parquet(INPUT_PATH)
df['timestamp'] = pd.to_datetime(df['timestamp'])
df = df.sort_values(['segment_key', 'timestamp']).reset_index(drop=True)
df[TARGET_COL] = pd.to_numeric(df[TARGET_COL], errors='coerce').fillna(0).clip(0, 5).astype(int)

print('Input shape:', df.shape)
print(df[TARGET_COL].value_counts().sort_index())

In [ ]:
# Prepare helper columns for duplicate detection
candidate_numeric_cols = [
    c for c in [
        'current_speed_kmh', 'traffic_index', 'delay_seconds', 'quality_flag',
        'speed_ratio', 'speed_delta', 'free_flow_speed_kmh',
        'time_sin', 'time_cos'
    ] if c in df.columns
]

def row_vector(row):
    return np.array([float(pd.to_numeric(row[c], errors='coerce')) for c in candidate_numeric_cols], dtype=np.float64)

def is_duplicate(row_vec, seen_vecs, mae_threshold=DUPLICATE_MAE_THRESHOLD):
    if not seen_vecs:
        return False
    for sv in seen_vecs:
        mae = float(np.mean(np.abs(row_vec - sv)))
        if mae <= mae_threshold:
            return True
    return False

def transition_flags(df_local):
    labels = df_local[TARGET_COL].to_numpy()
    segs = df_local['segment_key'].to_numpy()
    flags = np.zeros(len(df_local), dtype=bool)
    for i in range(1, len(df_local)):
        if segs[i] == segs[i-1] and labels[i] != labels[i-1]:
            flags[i] = True
            flags[i-1] = True
    return flags

In [ ]:
# ===== Stage 1: Undersampling majority classes 0,1,2 =====
counts = df[TARGET_COL].value_counts().to_dict()
M = int(counts.get(ANCHOR_CLASS, 0))
assert M > 0, 'Anchor class 3 không có dữ liệu.'

cap = int(CAP_MULTIPLIER * M)
base_probs = {}
for cls in MAJORITY_CLASSES:
    cls_count = int(counts.get(cls, 0))
    base_probs[cls] = min(1.0, cap / cls_count) if cls_count > 0 else 0.0

flags_transition = transition_flags(df)
seen_vecs = []
keep_idx = []

for i, row in df.iterrows():
    label = int(row[TARGET_COL])

    # Keep all class 3, 4, 5 in this stage
    if label not in MAJORITY_CLASSES:
        keep_idx.append(i)
        if candidate_numeric_cols:
            seen_vecs.append(row_vector(row))
        continue

    p = float(base_probs[label])
    rv = row_vector(row) if candidate_numeric_cols else np.array([], dtype=np.float64)

    dup = is_duplicate(rv, seen_vecs) if candidate_numeric_cols else False
    if dup:
        p = min(1.0, p * DUPLICATE_MULTIPLIER)
    if flags_transition[i]:
        p = min(1.0, p * TRANSITION_MULTIPLIER)

    if rng.random() < p:
        keep_idx.append(i)
        if candidate_numeric_cols:
            seen_vecs.append(rv)

df_stage1 = df.iloc[keep_idx].copy().reset_index(drop=True)
print('After stage1:', df_stage1.shape)
print(df_stage1[TARGET_COL].value_counts().sort_index())

In [ ]:
# ===== Stage 2: Keep anchor class 3 intact =====
before_class3 = int((df[TARGET_COL] == 3).sum())
after_class3 = int((df_stage1[TARGET_COL] == 3).sum())
assert before_class3 == after_class3, f'Class 3 changed: {before_class3} -> {after_class3}'
print('Class 3 preserved:', before_class3)

In [ ]:
# ===== Stage 3: Oversampling class 4 and 5 with CTGAN =====
continuous_cols = [
    c for c in [
        'current_speed_kmh', 'traffic_index', 'delay_seconds', 'quality_flag',
        'speed_ratio', 'speed_delta', 'free_flow_speed_kmh', 'default_lane_count',
        'time_sin', 'time_cos'
    ] if c in df_stage1.columns
]
discrete_cols = [c for c in ['segment_key', 'ward_district_id', 'tomtom_frc', 'is_one_way', 'day_of_week', 'shift_code', TARGET_COL] if c in df_stage1.columns]

df_min_4 = df_stage1[df_stage1[TARGET_COL] == 4].copy()
df_min_5 = df_stage1[df_stage1[TARGET_COL] == 5].copy()

def gaussian_augment(df_in, n_target, noise_pct=0.02, seed=42):
    if df_in.empty or n_target <= 0:
        return df_in.iloc[0:0].copy()
    rng_local = np.random.default_rng(seed)
    out = []
    take_idx = rng_local.integers(0, len(df_in), size=n_target)
    for idx in take_idx:
        row = df_in.iloc[int(idx)].copy()
        for c in continuous_cols:
            v = pd.to_numeric(row[c], errors='coerce')
            if pd.notna(v):
                row[c] = float(v) * (1.0 + rng_local.normal(0, noise_pct))
        out.append(row)
    return pd.DataFrame(out)

df_aug_5 = gaussian_augment(df_min_5, n_target=max(0, 2000 - len(df_min_5)), noise_pct=0.02, seed=SEED)
seed_5_for_ctgan = pd.concat([df_min_5, df_aug_5], ignore_index=True)

synthetic_4 = pd.DataFrame()
synthetic_5 = pd.DataFrame()

if USE_CTGAN:
    try:
        from sdv.single_table import CTGANSynthesizer

        if not df_min_4.empty:
            ctgan4 = CTGANSynthesizer(epochs=CTGAN_EPOCHS, verbose=True)
            ctgan4.fit(df_min_4)
            synthetic_4 = ctgan4.sample(num_rows=TARGET_SYNTHETIC_CLASS_4)

        if not seed_5_for_ctgan.empty:
            ctgan5 = CTGANSynthesizer(epochs=CTGAN_EPOCHS, verbose=True)
            ctgan5.fit(seed_5_for_ctgan)
            synthetic_5 = ctgan5.sample(num_rows=TARGET_SYNTHETIC_CLASS_5)

    except Exception as e:
        print('CTGAN unavailable, fallback to gaussian-only:', e)
        synthetic_4 = gaussian_augment(df_min_4, n_target=TARGET_SYNTHETIC_CLASS_4, noise_pct=0.02, seed=SEED+4)
        synthetic_5 = gaussian_augment(seed_5_for_ctgan, n_target=TARGET_SYNTHETIC_CLASS_5, noise_pct=0.02, seed=SEED+5)
else:
    synthetic_4 = gaussian_augment(df_min_4, n_target=TARGET_SYNTHETIC_CLASS_4, noise_pct=0.02, seed=SEED+4)
    synthetic_5 = gaussian_augment(seed_5_for_ctgan, n_target=TARGET_SYNTHETIC_CLASS_5, noise_pct=0.02, seed=SEED+5)

if not synthetic_4.empty:
    synthetic_4[TARGET_COL] = 4
if not synthetic_5.empty:
    synthetic_5[TARGET_COL] = 5

print('Synthetic class4:', len(synthetic_4))
print('Synthetic class5:', len(synthetic_5))

In [ ]:
# ===== Stage 4: Sanity check + merge + shuffle =====
def physics_sanity_check(df_in):
    w = df_in.copy()

    if 'current_speed_kmh' in w.columns:
        w = w[pd.to_numeric(w['current_speed_kmh'], errors='coerce').fillna(0) >= 0]

    if 'traffic_index' in w.columns:
        w = w[pd.to_numeric(w['traffic_index'], errors='coerce').fillna(0) >= 0]

    # Nếu tốc độ bằng 0 trong nhiều trường hợp thì congestion nên thuộc lớp cao
    if 'current_speed_kmh' in w.columns and TARGET_COL in w.columns:
        speed = pd.to_numeric(w['current_speed_kmh'], errors='coerce').fillna(0)
        label = pd.to_numeric(w[TARGET_COL], errors='coerce').fillna(0)
        inconsistent = (speed == 0) & (label <= 1)
        w = w[~inconsistent]

    return w.reset_index(drop=True)

syn_all = pd.concat([synthetic_4, synthetic_5], ignore_index=True)
syn_all = physics_sanity_check(syn_all)

df_balanced = pd.concat([df_stage1, syn_all], ignore_index=True)
df_balanced = df_balanced.sample(frac=1.0, random_state=SEED).reset_index(drop=True)

# Clean data types
df_balanced[TARGET_COL] = pd.to_numeric(df_balanced[TARGET_COL], errors='coerce').fillna(0).clip(0, 5).astype(int)
if 'timestamp' in df_balanced.columns:
    df_balanced['timestamp'] = pd.to_datetime(df_balanced['timestamp'])

df_balanced.to_parquet(OUTPUT_PATH, index=False)
print('Saved:', OUTPUT_PATH)
print('Final shape:', df_balanced.shape)
print('Final class counts:\n', df_balanced[TARGET_COL].value_counts().sort_index())